# IterCQR ROCC Pipeline

This notebook provides the end-to-end IterCQR baseline pipeline exposed by the ROCC API. It loads the selected TopiOCQA or QReCC split, generates IterCQR rewrites, retrieves with BM25 or ANCE, evaluates the resulting rankings, and stores aggregate, bootstrap, and per-query results. QReCC ANCE additionally supports shard-specific partial runs and their final merge.

ROCC refers here to the shared experiment API. No ROCC history selector is applied in this baseline pipeline.

### Repository setup

Use an existing ROCC checkout. Locally, open this notebook from the checkout; in Colab, first clone the repository over HTTPS (public) or with your own configured authentication (private), and set `ROCC_ROOT` to that directory. A mounted Drive checkout also works. This cell locates the checkout without copying keys, changing Git remotes, or pulling commits.


In [1]:
from pathlib import Path
import os
import sys

def resolve_project_root(start: Path) -> Path:
    configured = os.environ.get("ROCC_ROOT") or os.environ.get("MASTER_THESIS_PROJECT_ROOT")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if not (candidate / "experiments" / "rocc").is_dir():
            raise FileNotFoundError(f"Configured ROCC checkout is missing: {candidate}")
        return candidate
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "rocc").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from your ROCC checkout, or set ROCC_ROOT to it. "
        "In Colab, first clone the public repository over HTTPS or authenticate "
        "your own private checkout; no private clone or deploy key is assumed."
    )

PROJECT_ROOT = resolve_project_root(Path.cwd())
os.environ["MASTER_THESIS_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("checkout:", PROJECT_ROOT)


local runtime; GitHub checkout skipped


### Runtime setup

Locates the `experiments` directory, exposes the ROCC API, and initializes the local or Colab runtime.

In [2]:
from __future__ import annotations

import sys
from pathlib import Path

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

import pandas as pd
from IPython.display import display

from experiments import rocc

pd.set_option("display.max_colwidth", 180)

DATASET_NAME = "topiocqa"       # "topiocqa" | "qrecc"
SPLIT = "dev"                   # "train" | "dev" | "test"
RETRIEVER_KIND = "sparse"       # "sparse" | "dense"

# Match the dataset-specific baseline and final-retrieval configuration.
BM25_K1, BM25_B = {"topiocqa": (0.9, 0.4), "qrecc": (0.82, 0.68)}[DATASET_NAME]

RUN_ID = f"{DATASET_NAME}_{SPLIT}_{RETRIEVER_KIND}"
QRECC_SESSION_ID = "session_1"
QRECC_DENSE_MODE = "run_partial"  # "run_partial" | "merge_eval"

BATCH_SIZE = 16
TOP_K = 100
EVAL_KS = (3, 10, 100)
BOOTSTRAP_SAMPLES = 10_000
SAVE_PER_QUERY = True
MAX_BATCHES = None

print("experiments_dir:", EXPERIMENTS_DIR)
print("run_id:", RUN_ID)


experiments_dir: <project>/experiments
run_id: topiocqa_dev_sparse


### End-to-end pipeline execution

Runs the configured IterCQR workflow through the ROCC API: dataset loading, query rewriting, BM25 or ANCE retrieval, metric evaluation, bootstrap confidence intervals, and artifact persistence. For QReCC ANCE, the selected mode either creates one shard-specific partial ranking or merges and evaluates the completed session rankings.

In [3]:
result = rocc.run_itercqr_full_pipeline(
    dataset_name=DATASET_NAME,
    split=SPLIT,
    retriever_kind=RETRIEVER_KIND,
    bm25_k1=BM25_K1,
    bm25_b=BM25_B,
    run_id=RUN_ID,
    qrecc_session_id=QRECC_SESSION_ID,
    qrecc_dense_mode=QRECC_DENSE_MODE,
    batch_size=BATCH_SIZE,
    top_k=TOP_K,
    eval_ks=EVAL_KS,
    bootstrap_samples=BOOTSTRAP_SAMPLES,
    save_per_query=SAVE_PER_QUERY,
    max_batches=MAX_BATCHES,
    progress=True,
)

display(result.status)
display(result.data_summary)
display(result.index_summary)
display(result.capacity_summary)

if result.pipeline_result is not None:
    display(result.pipeline_result.retrievals.head())

if result.evaluation is not None:
    display(result.evaluation.aggregate)
    display(result.evaluation.bootstrap_summary)

display(pd.DataFrame([result.eval_result_paths]))


[1/8] Resolve paths and config
[2/8] Prepare resources and load dataset split


[3/8] Load IterCQR rewriter


The module name IterCQR Model (originally IterCQR Model) is not a valid Python identifier. Please rename the original module to avoid import issues.


[4/8] Prepare retriever index
[5/8] Load retriever


[6/8] Run rewrite + retrieval


...

,budget_tokens,include_history


,budget_tokens,include_history,sample_id,split,conv_id,turn_id,query,top_k,num_hits,top_docid,top_score,top_score_norm
0,512,True,dev:53:1,dev,53,1,"what was neighborhood watch if you are interested in learning more about the film ""Northern Watch"" by The Beatles or ""North",100,100,7144614,18.245100,1.0
1,512,True,dev:10:1,dev,10,1,"what was Quoll if you are interested in learning more about the species of animal known as Quoll, which is commonly found in the United",100,100,7466078,29.455200,1.0
2,512,True,dev:14:1,dev,14,1,"what was the Man Trap if you are interested in learning more about the film ""The Man Trap"" by The Beatles by The Beatles.",100,100,24045586,21.818899,1.0
3,512,True,dev:48:1,dev,48,1,"what is an oribi if you are interested in learning more about the term ""oribi"" used to refer to the body part that is commonly found",100,100,23843441,21.572399,1.0
4,512,True,dev:94:1,dev,94,1,"what is Hydrus if you are interested in learning more about the term ""hydrus"" used to refer to the plant that is commonly found in the",100,100,8375521,23.882200,1.0


,budget_tokens,include_history,metric,n,mean,variance
0,512,True,MRR,2514,0.164730,0.092874
1,512,True,nDCG@3,2514,0.150811,0.106559
2,512,True,R@3,2514,0.188146,0.152808
3,512,True,nDCG@10,2514,0.188421,0.107514
4,512,True,R@10,2514,0.293158,0.207299
5,512,True,nDCG@100,2514,0.236777,0.097113
6,512,True,R@100,2514,0.531026,0.249136


,budget_tokens,include_history,metric,bootstrap_samples,confidence,mean,variance,ci95_low,ci95_high
0,512,True,MRR,10000,0.95,0.164876,0.000036,0.153151,0.176796
1,512,True,nDCG@3,10000,0.95,0.150983,0.000042,0.138490,0.163587
2,512,True,R@3,10000,0.95,0.188359,0.000060,0.173031,0.203660
3,512,True,nDCG@10,10000,0.95,0.188568,0.000042,0.175896,0.201380
4,512,True,R@10,10000,0.95,0.293288,0.000081,0.275259,0.310670
5,512,True,nDCG@100,10000,0.95,0.236918,0.000038,0.224983,0.249122
6,512,True,R@100,10000,0.95,0.531150,0.000098,0.511933,0.550527


,evaluation_aggregate,bootstrap_summary,evaluation_per_query
0,/experiments/results/full_pipeline_runs/topiocqa_dev_sparse/evaluation_aggregate.csv,/experiments/results/full_pipeline_runs/topiocqa_dev_sparse/bootstrap_summary.csv,/experiments/results/full_pipeline_runs/topiocqa_dev_sparse/evaluation_per_query.csv


The saved baseline run covers all **2,514 TopiOCQA development queries** with route $I_{512}$. IterCQR uses the current query $q_t$ and recency-ordered history $H_t$ within total input budget $B=512$, with input length $n\leq B$. Each field has a 32-token limit, and greedy decoding uses maximum output sequence length $m_{\max}=32$. Retrieval depth is $k_{\mathrm{ret}}=100$ with BM25 parameters $k_1=0.9$ and $b=0.4$. The empty capacity summary is expected because GPU-capacity checks apply only to dense retrieval.

The thesis baseline reaches $\mathrm{MRR}=0.1647$, $\mathrm{nDCG}@3=0.1508$, $\mathrm{Recall}@10=0.2932$, and $\mathrm{Recall}@100=0.5310$. Query-level bootstrapping with $N_{\mathrm{boot}}=10{,}000$ resamples gives a 95% MRR confidence interval of **[0.1532, 0.1768]**. Here MRR is the mean reciprocal rank within the 100 retrieved passages. The code columns `R@10` and `R@100` denote the thesis metrics $\mathrm{Recall}@10$ and $\mathrm{Recall}@100$.

The displayed retrieval preview confirms that each rewrite produced 100 ranked passages. Aggregate metrics, bootstrap statistics, and results for all individual queries were persisted under `full_pipeline_runs/topiocqa_dev_sparse`. The recorded stage durations document pipeline execution and are not treated as a controlled latency benchmark.